# 🔬 AI-Based Prostate MRI Image Segmentation
## Using a Combined U-Net Deep Learning Model with Five Datasets

**Datasets**: I2CVB · MSD Task05 · NCI-ISBI 3T · NCI-ISBI Dx · PROSTATEx  
**Architecture**: Standard U-Net (PyTorch)  
**Task**: Whole-prostate binary segmentation from T2-weighted MRI  

---
### ▶️ How to Use This Notebook
1. Enable GPU: `Runtime → Change runtime type → T4 GPU`
2. Mount your Google Drive (contains raw datasets)
3. Run cells top-to-bottom (Shift+Enter)
4. Each section has detailed explanations
---

## 📦 Step 0 – Environment Setup & Library Installation

Install all required libraries not present in Colab's default environment.

In [ ]:
# Install medical imaging and utility libraries
!pip install -q nibabel SimpleITK pynrrd highdicom torchinfo
!pip install -q streamlit streamlit-option-menu
!pip install -q tabulate
print('✅ Libraries installed.')

In [ ]:
import torch
print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU             :', torch.cuda.get_device_name(0))
    print('GPU memory      :', torch.cuda.get_device_properties(0).total_memory // 1024**3, 'GB')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device    :', DEVICE)

## 📁 Step 1 – Mount Google Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# ── CONFIGURE THESE PATHS ────────────────────────────────────────────────────
DRIVE_ROOT       = Path('/content/drive/MyDrive/MRI_PROSTATE_UNET')
DATASET_ROOT     = DRIVE_ROOT / 'datasets'
PREPROCESSED_ROOT= DRIVE_ROOT / 'preprocessed'
RESULTS_ROOT     = DRIVE_ROOT / 'results'
# ─────────────────────────────────────────────────────────────────────────────

# Create directories
for d in [PREPROCESSED_ROOT, RESULTS_ROOT,
          RESULTS_ROOT/'graphs', RESULTS_ROOT/'predictions',
          RESULTS_ROOT/'models']:
    d.mkdir(parents=True, exist_ok=True)

# Clone project code to Colab runtime
!cp -r '{DRIVE_ROOT}'/*.py /content/ 2>/dev/null || echo 'Note: using inline code'

print('Drive mounted. Drive root:', DRIVE_ROOT)
print('Datasets   :', DATASET_ROOT)
print('Preprocessed:', PREPROCESSED_ROOT)

## 🏗️ Step 2 – U-Net Architecture

**Architecture**:
- Encoder: 4 levels with DoubleConv + MaxPool (channels: 32 → 64 → 128 → 256)
- Bottleneck: DoubleConv (512 channels)
- Decoder: 4 levels with TransposedConv + Skip connections
- Output: 1×1 Conv → Sigmoid → binary mask

**Loss**: Dice Loss + Binary Cross-Entropy (combined, α=0.5)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = DoubleConv(in_ch, out_ch)
        self.pool = nn.MaxPool2d(2)
    def forward(self, x):
        skip = self.conv(x)
        return self.pool(skip), skip


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = DoubleConv(out_ch * 2, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        return self.conv(torch.cat([skip, x], dim=1))


class UNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, enc=[32,64,128,256], bottleneck=512):
        super().__init__()
        self.enc1 = EncoderBlock(in_ch,    enc[0])
        self.enc2 = EncoderBlock(enc[0],   enc[1])
        self.enc3 = EncoderBlock(enc[1],   enc[2])
        self.enc4 = EncoderBlock(enc[2],   enc[3])
        self.bot  = DoubleConv(enc[3],     bottleneck)
        self.dec4 = DecoderBlock(bottleneck, enc[3])
        self.dec3 = DecoderBlock(enc[3],     enc[2])
        self.dec2 = DecoderBlock(enc[2],     enc[1])
        self.dec1 = DecoderBlock(enc[1],     enc[0])
        self.out  = nn.Conv2d(enc[0], out_ch, 1)

    def forward(self, x):
        x, s1 = self.enc1(x)
        x, s2 = self.enc2(x)
        x, s3 = self.enc3(x)
        x, s4 = self.enc4(x)
        x = self.bot(x)
        x = self.dec4(x, s4)
        x = self.dec3(x, s3)
        x = self.dec2(x, s2)
        x = self.dec1(x, s1)
        return self.out(x)

    def predict(self, x, thr=0.5):
        with torch.no_grad():
            return (torch.sigmoid(self.forward(x)) > thr).float()


# ── Test architecture ──────────────────────────────────────────────────────
model = UNet().to(DEVICE)
x     = torch.randn(2, 1, 256, 256).to(DEVICE)
out   = model(x)
print('Input :', x.shape)
print('Output:', out.shape)
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters    : {total:,}')
print(f'Trainable parameters: {trainable:,}')

## ⚖️ Step 3 – Loss Functions

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        p = probs.view(-1)
        t = targets.view(-1)
        inter = (p * t).sum()
        return 1 - (2*inter + self.smooth) / (p.sum() + t.sum() + self.smooth)


class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.5):
        super().__init__()
        self.alpha = alpha
        self.dice  = DiceLoss()
        self.bce   = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        return self.alpha * self.dice(logits, targets) + \
               (1 - self.alpha) * self.bce(logits, targets)


# Quick test
criterion = CombinedLoss()
y_pred = torch.randn(2, 1, 256, 256)
y_true = torch.randint(0, 2, (2, 1, 256, 256)).float()
loss = criterion(y_pred, y_true)
print(f'Combined Loss (random inputs): {loss.item():.4f}')

## 🔄 Step 4 – Dataset Preprocessing

Each dataset has a dedicated preprocessing function.  
All produce 256×256 normalised float32 images and binary uint8 masks saved as `.npy` files.

In [ ]:
import numpy as np
import nibabel as nib
import SimpleITK as sitk
import cv2
import json
import matplotlib.pyplot as plt
from pathlib import Path

TARGET_SIZE = (256, 256)
MIN_PROSTATE_PIXELS = 50

def normalize(vol):
    fg = vol[vol > 0]
    if fg.size == 0: return vol
    lo, hi = np.percentile(fg, [1, 99])
    vol = np.clip(vol, lo, hi)
    return ((vol - lo) / (hi - lo + 1e-7)).astype(np.float32)

def resize2d(arr, size, interp=cv2.INTER_LINEAR):
    return cv2.resize(arr.astype(np.float32), (size[1], size[0]), interpolation=interp).astype(arr.dtype)

def binarize(mask): return (mask > 0).astype(np.uint8)

def resample_mask(mask_path, ref_path):
    ref  = sitk.ReadImage(str(ref_path),  sitk.sitkFloat32)
    mask = sitk.ReadImage(str(mask_path), sitk.sitkUInt8)
    r = sitk.ResampleImageFilter()
    r.SetReferenceImage(ref)
    r.SetInterpolator(sitk.sitkNearestNeighbor)
    r.SetDefaultPixelValue(0)
    return sitk.GetArrayFromImage(r.Execute(mask)).astype(np.uint8)

def save_slices(vol, mask, out_dir, patient_id, context=True):
    out_dir = Path(out_dir) / patient_id
    out_dir.mkdir(parents=True, exist_ok=True)

    mask_bin = binarize(mask)
    vol_norm = normalize(vol)

    # Identify prostate slices (+ context)
    p_slices = [z for z in range(mask_bin.shape[0]) if mask_bin[z].sum() >= MIN_PROSTATE_PIXELS]
    ctx = set()
    for z in p_slices:
        for dz in [-1,0,1]:
            if 0 <= z+dz < mask_bin.shape[0]: ctx.add(z+dz)
    all_z = sorted(ctx) if context else p_slices

    saved = 0
    for z in all_z:
        img_s  = resize2d(vol_norm[z],           TARGET_SIZE, cv2.INTER_LINEAR)
        msk_s  = resize2d(mask_bin[z].astype(np.float32), TARGET_SIZE, cv2.INTER_NEAREST).astype(np.uint8)
        np.save(str(out_dir / f'image_slice_{z:04d}.npy'), img_s)
        np.save(str(out_dir / f'mask_slice_{z:04d}.npy'),  msk_s)
        saved += 1

    return saved, all_z, out_dir

print('✅ Preprocessing utilities defined.')

In [ ]:
# ── Run preprocessing for each dataset ──────────────────────────────────────
# Uncomment the dataset(s) you have downloaded.
# Replace paths with actual dataset locations.

import sys
sys.path.insert(0, str(DRIVE_ROOT))

from preprocessing.i2cvb_preprocessing import run_all as i2cvb_run
from preprocessing.msd_preprocessing import run_all as msd_run
from preprocessing.nci_3t_preprocessing import run_all as nci3t_run
from preprocessing.nci_dx_preprocessing import run_all as ncidx_run
from preprocessing.prostatex_preprocessing import run_all as prostatex_run

# I2CVB
# i2cvb_meta = i2cvb_run(str(DATASET_ROOT/'I2CVB'), str(PREPROCESSED_ROOT/'I2CVB'))

# MSD
# msd_meta = msd_run(str(DATASET_ROOT/'MSD/Task05_Prostate'), str(PREPROCESSED_ROOT/'MSD'))

# NCI-ISBI 3T
# nci3t_meta = nci3t_run(str(DATASET_ROOT/'NCI_ISBI_3T'), str(PREPROCESSED_ROOT/'NCI_ISBI_3T'))

# NCI-ISBI Dx
# ncidx_meta = ncidx_run(str(DATASET_ROOT/'NCI_ISBI_Dx'), str(PREPROCESSED_ROOT/'NCI_ISBI_Dx'))

# PROSTATEx
# prostatex_meta = prostatex_run(str(DATASET_ROOT/'PROSTATEx'), str(PREPROCESSED_ROOT/'PROSTATEx'))

print('Preprocessing step complete (or commented out). Verify verification.png files.')

## 📊 Step 5 – Dataset Discovery & Patient-Level Split

In [ ]:
import pandas as pd
import random

DATASET_NAMES = ['I2CVB', 'MSD', 'NCI_ISBI_3T', 'NCI_ISBI_Dx', 'PROSTATEx']

def discover_slices(preprocessed_root):
    records = []
    root = Path(preprocessed_root)
    for ds in DATASET_NAMES:
        ds_dir = root / ds
        if not ds_dir.exists():
            print(f'  ⚠ {ds} not found — skipping')
            continue
        for pdir in sorted(ds_dir.iterdir()):
            if not pdir.is_dir(): continue
            imgs = sorted(pdir.glob('image_slice_*.npy'))
            for img_f in imgs:
                mask_f = pdir / img_f.name.replace('image_', 'mask_')
                if mask_f.exists():
                    records.append({'dataset':pdir.parent.name,
                                    'patient_id':pdir.name,
                                    'image_path':str(img_f),
                                    'mask_path': str(mask_f)})
    return pd.DataFrame(records)

def patient_split(df, train=0.7, val=0.15, seed=42):
    rng = random.Random(seed)
    tr, vl, te = [], [], []
    for ds in df['dataset'].unique():
        sub = df[df['dataset']==ds]
        pts = sorted(sub['patient_id'].unique())
        rng.shuffle(pts)
        n = len(pts)
        n_tr = max(1, int(n*train))
        n_vl = max(1, int(n*val))
        tr.append(sub[sub['patient_id'].isin(pts[:n_tr])])
        vl.append(sub[sub['patient_id'].isin(pts[n_tr:n_tr+n_vl])])
        te.append(sub[sub['patient_id'].isin(pts[n_tr+n_vl:])])
        print(f'  {ds:20s}: total={n} train={n_tr} val={n_vl} test={n-n_tr-n_vl}')
    return (pd.concat(tr, ignore_index=True),
            pd.concat(vl, ignore_index=True),
            pd.concat(te, ignore_index=True))

df = discover_slices(PREPROCESSED_ROOT)
print(f'Total slices: {len(df)}  |  Datasets: {df["dataset"].nunique()}')
if not df.empty:
    display(df.groupby('dataset').size().rename('slices').to_frame())
    train_df, val_df, test_df = patient_split(df)
    print(f'\nTrain:{len(train_df)} Val:{len(val_df)} Test:{len(test_df)}')
    train_df.to_csv(RESULTS_ROOT/'split_train.csv', index=False)
    val_df.to_csv(  RESULTS_ROOT/'split_val.csv',   index=False)
    test_df.to_csv( RESULTS_ROOT/'split_test.csv',  index=False)
else:
    print('No preprocessed data found. Run preprocessing first.')

## 🚀 Step 6 – Model Training

**Configuration**: Adam · lr=0.001 · batch=8 · epochs=30 · Dice+BCE loss  
**Sampler**: Dataset-aware weighted sampler (balances smaller datasets)  
**Callbacks**: ReduceLROnPlateau · EarlyStopping (patience=6)

In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.optim as optim
import time
from collections import defaultdict

# ── Dataset ────────────────────────────────────────────────────────────────
class ProstateDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = np.load(row['image_path'])[np.newaxis].astype(np.float32)
        mask = np.load(row['mask_path'])[np.newaxis].astype(np.float32)
        if self.augment:
            if random.random()>0.5: img,mask=np.flip(img,2).copy(),np.flip(mask,2).copy()
            if random.random()>0.5: img,mask=np.flip(img,1).copy(),np.flip(mask,1).copy()
            k=random.randint(0,3); img=np.rot90(img,k,(1,2)).copy(); mask=np.rot90(mask,k,(1,2)).copy()
        return torch.from_numpy(img), torch.from_numpy(mask), row['dataset'], row['patient_id']

# ── Metrics ────────────────────────────────────────────────────────────────
def dice_batch(logits, targets, thr=0.5, eps=1e-7):
    p = (torch.sigmoid(logits)>thr).float()
    p=p.view(p.shape[0],-1); t=targets.view(targets.shape[0],-1)
    return ((2*(p*t).sum(1)+eps)/(p.sum(1)+t.sum(1)+eps)).mean().item()

# ── Training function ──────────────────────────────────────────────────────
def train_model(train_df, val_df, epochs=30, batch_size=8,
                lr=1e-3, patience=6, device='cuda'):
    # Datasets & loaders
    tr_ds = ProstateDataset(train_df, augment=True)
    vl_ds = ProstateDataset(val_df,   augment=False)

    # Weighted sampler
    ds_counts = train_df['dataset'].value_counts().to_dict()
    n_ds = len(ds_counts)
    wts = [n_ds / ds_counts[r['dataset']] for _, r in train_df.iterrows()]
    sampler = WeightedRandomSampler(wts, len(wts), replacement=True)

    tr_loader = DataLoader(tr_ds, batch_size=batch_size, sampler=sampler, num_workers=2)
    vl_loader = DataLoader(vl_ds, batch_size=batch_size, shuffle=False, num_workers=2)

    # Model, optimiser, loss
    model    = UNet().to(device)
    optim_   = optim.Adam(model.parameters(), lr=lr)
    criterion= CombinedLoss()
    sched    = ReduceLROnPlateau(optim_, 'min', factor=0.5, patience=3)
    scaler   = torch.cuda.amp.GradScaler() if device=='cuda' else None

    best_loss = float('inf')
    pat_cnt   = 0
    ckpt_path = RESULTS_ROOT/'models'/'combined_prostate_unet.pth'
    history   = defaultdict(list)

    for epoch in range(1, epochs+1):
        t0 = time.time()

        # Training
        model.train()
        tr_loss = tr_dice = 0
        for imgs, masks, _, _ in tr_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            optim_.zero_grad()
            if scaler:
                with torch.cuda.amp.autocast():
                    out  = model(imgs)
                    loss = criterion(out, masks)
                scaler.scale(loss).backward(); scaler.step(optim_); scaler.update()
            else:
                out  = model(imgs)
                loss = criterion(out, masks)
                loss.backward(); optim_.step()
            tr_loss += loss.item()
            tr_dice += dice_batch(out.detach(), masks)
        n_tr = max(len(tr_loader),1)
        tr_loss/=n_tr; tr_dice/=n_tr

        # Validation
        model.eval()
        vl_loss = vl_dice = 0
        with torch.no_grad():
            for imgs, masks, _, _ in vl_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                out = model(imgs)
                vl_loss += criterion(out, masks).item()
                vl_dice += dice_batch(out, masks)
        n_vl = max(len(vl_loader),1)
        vl_loss/=n_vl; vl_dice/=n_vl

        sched.step(vl_loss)
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_dice'].append(tr_dice)
        history['val_dice'].append(vl_dice)

        print(f'Ep {epoch:02d}/{epochs} | tr_loss={tr_loss:.4f} vl_loss={vl_loss:.4f} '
              f'tr_dice={tr_dice:.4f} vl_dice={vl_dice:.4f} [{time.time()-t0:.1f}s]')

        if vl_loss < best_loss:
            best_loss = vl_loss; pat_cnt = 0
            torch.save({'epoch':epoch,'model_state_dict':model.state_dict(),
                        'val_loss':vl_loss,'val_dice':vl_dice,'history':dict(history)}, ckpt_path)
            print(f'  ✅ Best model saved (val_loss={vl_loss:.4f})')
        else:
            pat_cnt += 1
            if pat_cnt >= patience:
                print(f'Early stopping at epoch {epoch}')
                break

    return model, dict(history), str(ckpt_path)

# ── Run training ───────────────────────────────────────────────────────────
if not df.empty:
    model, history, ckpt = train_model(
        train_df, val_df,
        epochs=30, batch_size=8, lr=1e-3, patience=6, device=DEVICE
    )
    print(f'\n✅ Training done. Checkpoint: {ckpt}')
else:
    print('No data available. Preprocess datasets first.')

## 📈 Step 7 – Training Visualisation

In [ ]:
if 'history' in dir():
    epochs_range = range(1, len(history['train_loss'])+1)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(epochs_range, history['train_loss'], 'b-o', ms=3, label='Train')
    axes[0].plot(epochs_range, history['val_loss'],   'r-o', ms=3, label='Val')
    axes[0].set_title('Training & Validation Loss'); axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(epochs_range, history['train_dice'], 'b-o', ms=3, label='Train')
    axes[1].plot(epochs_range, history['val_dice'],   'r-o', ms=3, label='Val')
    axes[1].set_title('Dice Score'); axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Dice'); axes[1].legend(); axes[1].grid(alpha=0.3)

    axes[2].plot(epochs_range, history['val_loss'], 'r-o', ms=3)
    axes[2].set_title('Validation Loss (detail)'); axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Loss'); axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(str(RESULTS_ROOT/'graphs'/'training_curves.png'), dpi=150)
    plt.show()
else:
    print('Run training first.')

## 📊 Step 8 – Model Evaluation (8 Metrics)

Metrics computed at **patient level** (3D volume) from test set predictions.

In [ ]:
import numpy as np

EPS = 1e-7
METRICS = ['Dice','IoU','Accuracy','Precision','Recall','F1','Sensitivity','Specificity']

def compute_metrics(pred, gt):
    p = pred.flatten().astype(bool)
    g = gt.flatten().astype(bool)
    TP = (p & g).sum();  TN = (~p & ~g).sum()
    FP = (p & ~g).sum(); FN = (~p & g).sum()
    dice  = (2*TP) / (2*TP+FP+FN+EPS)
    iou   = TP     / (TP+FP+FN+EPS)
    acc   = (TP+TN)/ (TP+TN+FP+FN+EPS)
    prec  = TP     / (TP+FP+EPS)
    rec   = TP     / (TP+FN+EPS)
    f1    = 2*prec*rec / (prec+rec+EPS)
    sens  = rec
    spec  = TN     / (TN+FP+EPS)
    return dict(Dice=dice, IoU=iou, Accuracy=acc, Precision=prec,
                Recall=rec, F1=f1, Sensitivity=sens, Specificity=spec)

@torch.no_grad()
def evaluate_model(model, test_df, device, thr=0.5):
    model.eval()
    records = []
    for (ds, pid), grp in test_df.groupby(['dataset','patient_id']):
        preds, gts = [], []
        for _, row in grp.iterrows():
            img = torch.from_numpy(np.load(row['image_path'])[None,None].astype(np.float32)).to(device)
            out = torch.sigmoid(model(img))[0,0].cpu().numpy()
            preds.append((out>thr).astype(np.uint8))
            gts.append(np.load(row['mask_path']).astype(np.uint8))
        pred_vol = np.stack(preds); gt_vol = np.stack(gts)
        m = compute_metrics(pred_vol, gt_vol)
        m.update({'dataset':ds, 'patient_id':pid})
        records.append(m)
    return pd.DataFrame(records)

if 'model' in dir() and not test_df.empty:
    results = evaluate_model(model, test_df, DEVICE)
    results.to_csv(RESULTS_ROOT/'metrics.csv', index=False)

    print('\n=== OVERALL PERFORMANCE ===')
    overall = results[METRICS].mean()
    for m, v in overall.items():
        print(f'  {m:<14}: {v:.4f}')

    print('\n=== DATASET-WISE PERFORMANCE ===')
    display(results.groupby('dataset')[METRICS].mean().round(4))
else:
    print('Run training and split steps first.')

## 🖼️ Step 9 – Performance Visualisation Graphs

In [ ]:
if 'results' in dir():
    ds_means = results.groupby('dataset')[METRICS].mean()
    overall_means = results[METRICS].mean()
    
    # 1. Eight-metric bar chart
    fig, ax = plt.subplots(figsize=(12, 5))
    colours = plt.cm.Set2(np.linspace(0,1,len(METRICS)))
    bars = ax.bar(METRICS, overall_means.values, color=colours, edgecolor='black', lw=0.5)
    for bar, val in zip(bars, overall_means.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_ylim(0, 1.1); ax.set_ylabel('Score'); ax.set_title('Overall Model Performance – All 8 Metrics')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(RESULTS_ROOT/'graphs'/'eight_metrics.png'), dpi=150)
    plt.show()

    # 2. Dataset comparison – Dice, IoU, Accuracy
    datasets = list(ds_means.index)
    x = np.arange(len(datasets)); w = 0.25
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(x-w,   ds_means['Dice'],     w, label='Dice',     color='steelblue')
    ax.bar(x,     ds_means['IoU'],      w, label='IoU',      color='coral')
    ax.bar(x+w,   ds_means['Accuracy'], w, label='Accuracy', color='seagreen')
    ax.set_xticks(x); ax.set_xticklabels(datasets, rotation=10)
    ax.set_ylim(0,1.1); ax.legend(); ax.set_title('Dataset-wise Dice / IoU / Accuracy')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(RESULTS_ROOT/'graphs'/'dataset_comparison.png'), dpi=150)
    plt.show()

    # 3. Precision & Recall
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x-w/2, ds_means['Precision'], w, label='Precision', color='royalblue')
    ax.bar(x+w/2, ds_means['Recall'],    w, label='Recall',    color='tomato')
    ax.set_xticks(x); ax.set_xticklabels(datasets, rotation=10)
    ax.set_ylim(0,1.1); ax.legend(); ax.set_title('Precision & Recall per Dataset')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(RESULTS_ROOT/'graphs'/'precision_recall.png'), dpi=150)
    plt.show()

    # 4. Five-dataset comparison table
    print('\n=== FIVE-DATASET COMPARISON TABLE ===')
    display(ds_means.round(4).style.background_gradient(cmap='YlGn', axis=0))
else:
    print('Run evaluation step first.')

## 🔍 Step 10 – Sample Prediction Visualisation

In [ ]:
@torch.no_grad()
def show_predictions(model, test_df, device, n_samples=3, thr=0.5):
    model.eval()
    samples = test_df.sample(min(n_samples, len(test_df)), random_state=42)
    fig, axes = plt.subplots(len(samples), 4, figsize=(16, 4*len(samples)))
    if len(samples) == 1: axes = axes[np.newaxis,:]

    for r, (_, row) in enumerate(samples.iterrows()):
        img_np  = np.load(row['image_path']).astype(np.float32)
        gt_np   = np.load(row['mask_path']).astype(np.uint8)
        img_t   = torch.from_numpy(img_np[None,None]).to(device)
        probs   = torch.sigmoid(model(img_t))[0,0].cpu().numpy()
        pred_np = (probs > thr).astype(np.uint8)

        overlay = np.stack([img_np]*3, -1)
        overlay[gt_np==1, 1] = 1.0       # GT = green
        overlay[pred_np==1, 0] = 1.0     # Pred = red

        axes[r,0].imshow(img_np, cmap='gray'); axes[r,0].set_title(f'T2w MRI [{row["dataset"]}]')
        axes[r,1].imshow(gt_np,  cmap='gray'); axes[r,1].set_title('Ground Truth')
        axes[r,2].imshow(pred_np,cmap='gray'); axes[r,2].set_title('Prediction')
        axes[r,3].imshow(overlay);             axes[r,3].set_title('Overlay (GT=G, Pred=R)')
        for ax in axes[r]: ax.axis('off')

    plt.tight_layout()
    plt.savefig(str(RESULTS_ROOT/'predictions'/'sample_predictions.png'), dpi=150)
    plt.show()

if 'model' in dir() and not test_df.empty:
    show_predictions(model, test_df, DEVICE)
else:
    print('Run training first.')

## 📤 Step 11 – Predict on New MRI Image (Upload)

In [ ]:
from google.colab import files

print('Upload a prostate MRI slice (.npy, .png, or .jpg):')
uploaded = files.upload()

for fname, data in uploaded.items():
    import io
    if fname.endswith('.npy'):
        arr = np.load(io.BytesIO(data)).astype(np.float32)
        if arr.ndim == 3: arr = arr[arr.shape[0]//2]
    else:
        from PIL import Image
        arr = np.array(Image.open(io.BytesIO(data)).convert('L'), dtype=np.float32)

    arr = cv2.resize(arr, (256,256), interpolation=cv2.INTER_LINEAR)
    lo, hi = np.percentile(arr,1), np.percentile(arr,99)
    if hi>lo: arr = (np.clip(arr,lo,hi)-lo)/(hi-lo)

    if 'model' in dir():
        with torch.no_grad():
            img_t = torch.from_numpy(arr[None,None]).to(DEVICE)
            probs = torch.sigmoid(model(img_t))[0,0].cpu().numpy()
            pred  = (probs > 0.5).astype(np.uint8)

        overlay = np.stack([arr]*3,-1)
        overlay[pred==1,0]=1.0; overlay[pred==1,1]=0.3; overlay[pred==1,2]=0.0

        fig, axes = plt.subplots(1,3,figsize=(12,4))
        axes[0].imshow(arr,     cmap='gray'); axes[0].set_title('Input MRI')
        axes[1].imshow(pred,    cmap='gray'); axes[1].set_title('Predicted Mask')
        axes[2].imshow(overlay);              axes[2].set_title('Overlay')
        for ax in axes: ax.axis('off')
        plt.suptitle(f'Uploaded: {fname}', fontsize=11)
        plt.tight_layout(); plt.show()
    else:
        print('Model not trained yet. Run training step first.')

## ✅ Step 12 – Summary & Download Results

In [ ]:
import zipfile

print('='*60)
print(' PROJECT SUMMARY')
print('='*60)
print(' Architecture : Standard U-Net (PyTorch)')
print(' Input size   : 256×256×1 (grayscale MRI)')
print(' Datasets     : I2CVB, MSD, NCI-ISBI 3T/Dx, PROSTATEx')
print(' Task         : Binary whole-prostate segmentation')
print(' Loss         : Dice + BCE (α=0.5)')
print(' Optimizer    : Adam (lr=0.001)')
if 'results' in dir():
    ov = results[METRICS].mean()
    print(f' Best Val Dice: {history["val_dice"][-1]:.4f}')
    print('\n Results (mean over test set):')
    for m in METRICS:
        print(f'  {m:<14}: {ov[m]:.4f}')
print('='*60)

# Zip results for download
zip_path = '/content/prostate_results.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for f in RESULTS_ROOT.rglob('*'):
        if f.is_file():
            zf.write(f, f.relative_to(RESULTS_ROOT.parent))

print(f'\nResults zipped → {zip_path}')
files.download(zip_path)